# Homework: Forecasting Ocean Waves with an LSTM

**Dataset:** NOAA National Data Buoy Center station 41004 (Edisto, offshore South Carolina)  
**Task:** Use the previous 24 hours of ocean and weather observations to predict significant wave height (`WVHT`) three hours ahead.  
**Models:** Persistence, ridge regression, a multilayer perceptron (MLP), and a long short-term memory network (LSTM).

This assignment asks whether preserving the temporal order of a multivariate environmental sequence improves prediction. A complicated model is useful only if it improves upon honest, simpler baselines.

## Learning goals

By the end of this assignment, you should be able to:

1. Convert multivariate buoy observations into supervised sequences.
2. Explain why time-series data require temporal rather than random validation.
3. Fit preprocessing using past observations only.
4. Compare persistence, linear, non-sequential neural-network, and LSTM forecasts.
5. Use expanding-window forecast origins for validation and final testing.
6. Diagnose model performance during both calm and high-wave conditions.

## Submission

Submit the completed notebook, all requested figures and tables, and written responses to the Final Questions.


In [ ]:
# Install once if needed:
# %pip install numpy pandas matplotlib seaborn scikit-learn tensorflow

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Reduce TensorFlow's routine startup messages.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

def reset_seeds(seed=SEED):
    # Reset all seeds before constructing each neural network.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

reset_seeds()
sns.set_theme(style="whitegrid")


## Load NOAA buoy observations

Station 41004 is approximately 41 nautical miles southeast of Charleston, South Carolina. NOAA provides annual compressed standard-meteorological files. We will use 2018–2024. The files have already been downloaded and are stored on GitHub.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/noaa_wave_height.csv"
hourly = pd.read_csv(DATA_URL, index_col=0, parse_dates=[0])
                    
feature_names = [
    "WVHT", "WSPD", "DPD", "APD", "PRES", "ATMP", "WTMP",
    "mwd_sin", "mwd_cos", "hour_sin", "hour_cos", "year_sin", "year_cos",
]

print("Date range:", hourly.index.min(), "to", hourly.index.max())
print("Hourly rows:", len(hourly))
display(hourly[feature_names + ["target_wvht"]].isna().mean().to_frame("missing_fraction"))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
hourly.loc["2024", "target_wvht"].plot(ax=axes[0], color="navy")
axes[0].set_ylabel("WVHT (m)")
axes[0].set_title("Buoy 41004 observations during 2024")

hourly.loc["2024", "WSPD"].plot(ax=axes[1], color="darkorange")
axes[1].set_ylabel("Wind speed (m/s)")

hourly.loc["2024", "PRES"].plot(ax=axes[2], color="darkgreen")
axes[2].set_ylabel("Pressure (hPa)")
axes[2].set_xlabel("Time")
plt.tight_layout()
plt.show()


## Convert the record into supervised sequences

Each sample uses 24 hourly rows ending at forecast origin (t) and predicts observed significant wave height at (t+3) hours. A persistence forecast predicts that the future wave height will equal the most recently observed wave height.


In [ ]:
LOOKBACK = 24
HORIZON = 3

def make_sequences(frame, features, lookback=24, horizon=3):
    X, y, metadata = [], [], []
    values = frame[features].to_numpy(dtype=np.float32)
    target = frame["target_wvht"].to_numpy(dtype=np.float32)
    times = frame.index

    for end_idx in range(lookback - 1, len(frame) - horizon):
        start_idx = end_idx - lookback + 1
        target_idx = end_idx + horizon
        window = values[start_idx : end_idx + 1]

        # Keep only complete input windows with an observed future target.
        if np.isnan(window).any() or np.isnan(target[target_idx]):
            continue

        X.append(window)
        y.append(target[target_idx])
        metadata.append(
            {
                "origin_time": times[end_idx],
                "target_time": times[target_idx],
                "persistence": values[end_idx, features.index("WVHT")],
            }
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(metadata),
    )

X_all, y_all, sequence_info = make_sequences(
    hourly,
    feature_names,
    lookback=LOOKBACK,
    horizon=HORIZON,
)

print("X shape:", X_all.shape, "(samples, hours, features)")
print("y shape:", y_all.shape)
display(sequence_info.head())


## Define expanding forecast origins

Each forecast origin uses all eligible samples available before the forecast month and predicts the following month.

For manageable homework runtime, the default uses January, April, July, and October forecast months.

- **2023 origins:** validation and model selection
- **2024 origins:** untouched final testing


In [ ]:
origin_months = [1, 4, 7, 10]

validation_origins = [pd.Timestamp(2023, month, 1) for month in origin_months]
test_origins = [pd.Timestamp(2024, month, 1) for month in origin_months]

def month_end_exclusive(origin):
    return origin + pd.offsets.MonthBegin(1)

for origin in validation_origins:
    print(
        f"Train through {origin - pd.Timedelta(hours=1):%Y-%m-%d %H:%M}; "
        f"forecast {origin:%Y-%m}"
    )


## Prepare one leakage-safe forecast origin

The helper below:

1. Selects training samples whose targets occur before the forecast month.
2. Reserves the final 30 days of that historical record for early stopping.
3. Fits feature and target scalers using the remaining training data only.
4. Applies those scalers to internal-validation and forecast samples.
5. Limits training to the most recent 30,000 sequences for practical runtime.


In [ ]:
MAX_TRAIN_SAMPLES = 30000
INNER_VALIDATION_DAYS = 30

def prepare_origin(origin):
    forecast_end = month_end_exclusive(origin)

    train_mask = sequence_info["target_time"] < origin
    forecast_mask = sequence_info["target_time"].between(
        origin, forecast_end, inclusive="left"
    )

    train_idx = np.flatnonzero(train_mask.to_numpy())
    forecast_idx = np.flatnonzero(forecast_mask.to_numpy())
    if len(train_idx) == 0 or len(forecast_idx) == 0:
        raise ValueError(f"Insufficient data for forecast origin {origin}")

    if len(train_idx) > MAX_TRAIN_SAMPLES:
        train_idx = train_idx[-MAX_TRAIN_SAMPLES:]

    inner_cutoff = origin - pd.Timedelta(days=INNER_VALIDATION_DAYS)
    inner_val_mask = sequence_info.iloc[train_idx]["target_time"] >= inner_cutoff
    fit_idx = train_idx[~inner_val_mask.to_numpy()]
    inner_idx = train_idx[inner_val_mask.to_numpy()]

    feature_scaler = StandardScaler()
    feature_scaler.fit(X_all[fit_idx].reshape(-1, X_all.shape[-1]))

    target_scaler = StandardScaler()
    target_scaler.fit(y_all[fit_idx].reshape(-1, 1))

    def transform_X(indices):
        shape = X_all[indices].shape
        return feature_scaler.transform(
            X_all[indices].reshape(-1, shape[-1])
        ).reshape(shape).astype(np.float32)

    def transform_y(indices):
        return target_scaler.transform(
            y_all[indices].reshape(-1, 1)
        ).ravel().astype(np.float32)

    return {
        "X_fit": transform_X(fit_idx),
        "y_fit": transform_y(fit_idx),
        "X_inner": transform_X(inner_idx),
        "y_inner": transform_y(inner_idx),
        "X_forecast": transform_X(forecast_idx),
        "y_forecast": y_all[forecast_idx],
        "forecast_info": sequence_info.iloc[forecast_idx].reset_index(drop=True),
        "target_scaler": target_scaler,
    }

example_fold = prepare_origin(validation_origins[0])
for key in ["X_fit", "X_inner", "X_forecast"]:
    print(key, example_fold[key].shape)


## Build the comparison models

Ridge regression receives a flattened 24 × feature input. The MLP also receives a flattened input, allowing nonlinear relationships but discarding the explicit sequence organization. The LSTM receives a 24 × feature sequence.


In [ ]:
def build_mlp(input_shape, hidden_units=32, dropout=0.10, learning_rate=0.001):
    model = keras.Sequential(
        [
            layers.Input(shape=input_shape),
            layers.Flatten(),
            layers.Dense(hidden_units, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(1),
        ],
        name="wave_mlp",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
    )
    return model


def build_lstm(input_shape, hidden_units=32, dropout=0.10, learning_rate=0.001):
    model = keras.Sequential(
        [
            layers.Input(shape=input_shape),
            layers.LSTM(hidden_units),
            layers.Dropout(dropout),
            layers.Dense(1),
        ],
        name="wave_lstm",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
    )
    return model

## Fit and evaluate one forecast origin

In [ ]:
def regression_metrics(observed, predicted):
    return {
        "rmse": mean_squared_error(observed, predicted) ** 0.5,
        "mae": mean_absolute_error(observed, predicted),
    }

def evaluate_origin(origin, settings, verbose=0):
    fold = prepare_origin(origin)
    predictions = pd.DataFrame({
        "target_time": fold["forecast_info"]["target_time"],
        "observed": fold["y_forecast"],
        "Persistence": fold["forecast_info"]["persistence"],
    })

    # Ridge uses the same scaled inputs and target as the neural networks.
    ridge = Ridge(alpha=settings["ridge_alpha"])
    ridge.fit(
        fold["X_fit"].reshape(len(fold["X_fit"]), -1),
        fold["y_fit"],
    )
    ridge_scaled = ridge.predict(
        fold["X_forecast"].reshape(len(fold["X_forecast"]), -1)
    )
    predictions["Ridge"] = fold["target_scaler"].inverse_transform(
        ridge_scaled.reshape(-1, 1)
    ).ravel()

    def early_stopping():
        return keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=settings["patience"],
            restore_best_weights=True,
        )

    tf.keras.backend.clear_session()
    reset_seeds()
    mlp = build_mlp(
        fold["X_fit"].shape[1:],
        hidden_units=settings["hidden_units"],
        dropout=settings["dropout"],
        learning_rate=settings["learning_rate"],
    )
    mlp_history = mlp.fit(
        fold["X_fit"], fold["y_fit"],
        validation_data=(fold["X_inner"], fold["y_inner"]),
        epochs=settings["max_epochs"],
        batch_size=settings["batch_size"],
        shuffle=True,
        callbacks=[early_stopping()],
        verbose=verbose,
    )
    mlp_scaled = mlp.predict(fold["X_forecast"], verbose=0).ravel()
    predictions["MLP"] = fold["target_scaler"].inverse_transform(
        mlp_scaled.reshape(-1, 1)
    ).ravel()

    tf.keras.backend.clear_session()
    reset_seeds()
    lstm = build_lstm(
        fold["X_fit"].shape[1:],
        hidden_units=settings["hidden_units"],
        dropout=settings["dropout"],
        learning_rate=settings["learning_rate"],
    )
    lstm_history = lstm.fit(
        fold["X_fit"], fold["y_fit"],
        validation_data=(fold["X_inner"], fold["y_inner"]),
        epochs=settings["max_epochs"],
        batch_size=settings["batch_size"],
        shuffle=True,
        callbacks=[early_stopping()],
        verbose=verbose,
    )
    lstm_scaled = lstm.predict(fold["X_forecast"], verbose=0).ravel()
    predictions["LSTM"] = fold["target_scaler"].inverse_transform(
        lstm_scaled.reshape(-1, 1)
    ).ravel()

    predictions.attrs["mlp_history"] = mlp_history.history
    predictions.attrs["lstm_history"] = lstm_history.history
    return predictions


default_settings = {
    "ridge_alpha": 10.0,
    "hidden_units": 32,
    "dropout": 0.10,
    "learning_rate": 0.001,
    "batch_size": 64,
    "max_epochs": 60,
    "patience": 8,
}


## Expanding-window validation

Run the model at each 2023 origin, combine the forecasts, and calculate overall metrics.

In [ ]:
def run_origins(origins, settings):
    outputs = []
    for i, origin in enumerate(origins):
        print(f"Origin {i + 1}/{len(origins)}: {origin:%Y-%m}")
        result = evaluate_origin(origin, settings)
        result.insert(0, "origin", origin)
        outputs.append(result)
        tf.keras.backend.clear_session()
    return pd.concat(outputs, ignore_index=True)


# Run the validation origins after completing evaluate_origin().
validation_predictions = run_origins(validation_origins, default_settings)

In [ ]:
def summarize_predictions(predictions):
    model_columns = [
        c for c in predictions.columns
        if c not in ["origin", "target_time", "observed"]
    ]
    rows = []
    for model in model_columns:
        rows.append({
            "model": model,
            **regression_metrics(predictions["observed"], predictions[model]),
        })
    return pd.DataFrame(rows).sort_values("rmse")


validation_metrics = summarize_predictions(validation_predictions)
display(validation_metrics)

validation_long = validation_predictions.melt(
    id_vars=["origin", "target_time", "observed"],
    value_vars=["Persistence", "Ridge", "MLP", "LSTM"],
    var_name="model",
    value_name="predicted",
)
validation_by_origin_rows = []
for (origin, model), group in validation_long.groupby(["origin", "model"]):
    validation_by_origin_rows.append({
        "origin": origin,
        "model": model,
        **regression_metrics(group["observed"], group["predicted"]),
    })
validation_by_origin = pd.DataFrame(validation_by_origin_rows)
display(validation_by_origin)

## Controlled LSTM experiments

Change **one setting at a time** and evaluate only on the 2023 origins. Record each experiment in a table.

1. Hidden units: 8 versus 32
2. Lookback: 12, 24, or 48 hours (rebuild sequences when this changes)
3. Dropout: 0.0 versus 0.20
4. Learning rate: 0.0003 versus 0.001
5. Forecast horizon: 3 or 6 hours
6. ***TODO***: One of your choosing

In [ ]:
def evaluate_configuration(label, settings, lookback=24, horizon=3):
    global X_all, y_all, sequence_info

    saved = (X_all, y_all, sequence_info)
    try:
        X_all, y_all, sequence_info = make_sequences(
            hourly, feature_names, lookback=lookback, horizon=horizon
        )
        predictions = run_origins(validation_origins, settings)
        metrics = summarize_predictions(predictions)
        lstm_row = metrics.loc[metrics["model"] == "LSTM"].iloc[0]
        return {
            "experiment": label,
            "lookback": lookback,
            "horizon": horizon,
            "hidden_units": settings["hidden_units"],
            "dropout": settings["dropout"],
            "learning_rate": settings["learning_rate"],
            "lstm_rmse": lstm_row["rmse"],
            "lstm_mae": lstm_row["mae"],
        }
    finally:
        X_all, y_all, sequence_info = saved


experiment_specs = []

settings_8 = default_settings.copy()
settings_8["hidden_units"] = 8
experiment_specs.append(("8 hidden units", settings_8, 24, 3))

settings_dropout = default_settings.copy()
settings_dropout["dropout"] = 0.20
experiment_specs.append(("dropout 0.20", settings_dropout, 24, 3))

settings_lr = default_settings.copy()
settings_lr["learning_rate"] = 0.0003
experiment_specs.append(("learning rate 0.0003", settings_lr, 24, 3))

settings_trial = # TODO: default_settings.copy()
settings_trial["?????"] = # TODO
experiment_specs.append(("YOUR NAME HERE", settings_lr, 24, 3))

experiment_specs.append(("6-hour horizon", default_settings.copy(), 24, 6))
experiment_specs.append(("lookback 48-hours", default_settings.copy(), 48, 3))

experiment_results = pd.DataFrame(
    [evaluate_configuration(*spec) for spec in experiment_specs]
)
display(experiment_results.sort_values("lstm_rmse"))

## Final expanding-window test

Enter the settings selected from 2023 validation. Do not change them after looking at the 2024 results.

In [ ]:
final_settings = default_settings.copy()

test_predictions = run_origins(test_origins, final_settings)
test_metrics = summarize_predictions(test_predictions)
display(test_metrics)

In [ ]:
test_long = test_predictions.melt(
    id_vars=["origin", "target_time"],
    value_vars=["observed", "Persistence", "Ridge", "MLP", "LSTM"],
    var_name="series",
    value_name="wave_height_m",
)

g = sns.relplot(
    data=test_long,
    x="target_time",
    y="wave_height_m",
    hue="series",
    row="origin",
    kind="line",
    height=2.3,
    aspect=4.2,
    facet_kws={"sharex": False, "sharey": True},
)
g.set_axis_labels("Time", "Significant wave height (m)")
g.fig.suptitle("Three-hour wave forecasts at expanding 2024 origins", y=1.01)
plt.show()

## Diagnose calm and high-wave performance

Overall averages can hide poor performance during hazardous conditions. This block compares calm/moderate and high-wave errors, defined using the 90th percentile of historical pre-2024 observed wave height.

In [ ]:
# Use historical information only to define the threshold.
historical_index = sequence_info.loc[
    sequence_info["target_time"] < pd.Timestamp("2024-01-01"),
].index
high_wave_threshold = np.quantile(y_all[historical_index], 0.90)
print("Historical 90th-percentile threshold:", high_wave_threshold, "m")

regime_predictions = test_predictions.copy()
regime_predictions["wave_regime"] = np.where(
    regime_predictions["observed"] >= high_wave_threshold,
    "High wave",
    "Calm/moderate",
)

regime_rows = []
for regime, group in regime_predictions.groupby("wave_regime"):
    for model in ["Persistence", "Ridge", "MLP", "LSTM"]:
        regime_rows.append({
            "wave_regime": regime,
            "model": model,
            "n": len(group),
            **regression_metrics(group["observed"], group[model]),
        })

regime_metrics = pd.DataFrame(regime_rows).sort_values(["wave_regime", "rmse"])
display(regime_metrics)

In [ ]:
residual_data = test_predictions.copy()
residual_data["residual"] = residual_data["observed"] - residual_data["LSTM"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(
    data=residual_data,
    x="target_time",
    y="residual",
    hue="origin",
    palette="viridis",
    legend=False,
    ax=axes[0],
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("LSTM residuals through time")
axes[0].set_xlabel("Target time")
axes[0].set_ylabel("Observed - predicted WVHT (m)")

sns.scatterplot(
    data=residual_data,
    x="observed",
    y="residual",
    alpha=0.55,
    ax=axes[1],
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].axvline(high_wave_threshold, color="red", linestyle="--", label="90th percentile")
axes[1].set_title("Residuals versus observed wave height")
axes[1].set_xlabel("Observed WVHT (m)")
axes[1].set_ylabel("Observed - predicted WVHT (m)")
axes[1].legend()
plt.tight_layout()
plt.show()

## Final Questions

Answer the following:

1. What does each of the three dimensions of `X_all` represent?
2. Why is the target located three rows after the final input row?
3. What information does persistence use, and what potentially useful information does it ignore?
4. Why can complete sequence samples be shuffled while the 24 timesteps inside each sequence must remain ordered?
5. What does `restore_best_weights=True` accomplish?
6. Which model has the lowest validation RMSE and MAE during validation? Does the LSTM improve upon persistence?
7. Does the LSTM improve upon the MLP? What does that comparison suggest about preserving sequence structure?
8. Which model performs best on the untouched 2024 test origins? Does the model ranking agree with the 2023 validation ranking?
9. Does the best overall model remain best during high-wave conditions?
10. Does the LSTM systematically underpredict the largest waves?
11. Why can a small overall RMSE conceal an important operational weakness?
12. Was the LSTM appropriate for this marine forecasting task?